In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../shared'))   # repo shared/ (paths.py); run this notebook from its folder
from paths import outdir as _outdir, neuropixels, NEUROPIXELS_CACHE
NP_DATA = os.path.join(_outdir('fig5_neuropixels/notebooks/data'), '')   # was 'data/' (the data folder of ../notebooks/script.ipynb)
NP_SAVE = os.path.join(_outdir('fig5_neuropixels/notebooks/save'), '')   # was 'save/' and D:\...\mice-aibs\save\ (CITS results)
NP_FIGS = os.path.join(_outdir('fig5_neuropixels/legacy/figs'), '')   # was 'figs/'
NP_SAVEDIR = _outdir('fig5_neuropixels/notebooks/savedir')   # was D:\OneDrive - UW\research\savedir
from cits.methods import cits_full_weighted #RUN
import multiprocessing
import numpy as np
import time
import os
import pickle as pkl
import matplotlib.pyplot as plt

In [ ]:
#USE THIS FOR UNITS STUFF
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
from collections import defaultdict
import matplotlib.patches as mpatches

import os
import numpy as np
import pickle as pkl
from collections import defaultdict
from neuropixels_testresults import convert_fcmat2units

from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache
data_directory = NP_DATA
save_directory = NP_SAVE
manifest_path = os.path.join(NEUROPIXELS_CACHE, "manifest.json")
cache = EcephysProjectCache.from_warehouse(manifest=manifest_path)
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
sess_id = 791319847

#stim_label=stim_label_list[0]
class A_Config:
    def __init__(self, save_dir, sess_id, stimulus):
        
        #self.data_dir = data_dir
        self.save_dir = save_dir
        self.sess_id = sess_id
        self.stimulus = stimulus

        self.name = "{}/ID{}_{}".format(self.save_dir, self.sess_id, self.stimulus)
#allencon = A_Config(data_directory,sess_id,stim_label)
#X_load=pkl.load(open(allencon.name+'_bin_{}_X_idx-{}.p'.format(bin_size,idx),'rb'))
#X_load.shape
#%% units labeled
data = cache.get_session_data(sess_id)
# Get all 555 neuron structure labels
unit_labels_full = data.units['ecephys_structure_acronym'].values  # length 555

units_to_use_idx_={}
for stim_label in stim_label_list:
    allencon = A_Config(data_directory,sess_id,stim_label)
    units_to_use_idx_[stim_label] = np.where(pkl.load(open(allencon.name+'_units2use_stim_{}.p'.format(stim_label),'rb')))[0]

setnow = [set(unit) for unit in units_to_use_idx_.values()]
setnow = set.union(*setnow)
setnow = sorted(list(setnow))
units = setnow
unit_out = data.units.iloc[units,:]
unit_labels = unit_out['ecephys_structure_acronym'].values
labels_ordered = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam', 'CA1', 'CA2', 'CA3', 'DG', 'SUB', 'POL', 'LGv', 'LP']
permute = []
for label in labels_ordered:
    permute = permute + list(np.where(unit_labels==label)[0])
unit_labels_permuted = unit_labels[permute]


In [ ]:
# --- FIRST PASS: Compute global weight min/max and store edge data ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])  # NEW: Take absolute value
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)

In [ ]:
import os
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from collections import defaultdict

# CONFIGURATION
save_directory = NP_SAVE
plot_save_directory = NP_FIGS
os.makedirs(plot_save_directory, exist_ok=True)

sess_id              = 791319847
stim_label_list      = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha, bin_size, niter = 0.05, 0.01, 50
min_stability        = 0.8

# Color maps
visual_cortex = ['VISp','VISl','VISrl','VISal','VISpm','VISam']
thalamus      = ['LGv','LP']
hippocampus   = ['CA1','CA2','CA3','DG','SUB']
other         = ['POL']

def get_shaded_colors(regions, cmap_name):
    cmap   = cm.get_cmap(cmap_name)
    shades = np.linspace(0.3, 0.9, len(regions))
    return {r: cmap(shades[i]) for i,r in enumerate(regions)}

color_map = {}
color_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_map.update(get_shaded_colors(thalamus,      'Greens'))
color_map.update(get_shaded_colors(hippocampus,   'Reds'))
color_map.update(get_shaded_colors(other,         'Purples'))

cmap10 = plt.get_cmap('tab10')

# PERMUTE BY REGION
broad_map = {r:'Visual cortex' for r in visual_cortex}
broad_map.update({r:'Thalamus'    for r in thalamus})
broad_map.update({r:'Hippocampus' for r in hippocampus})
broad_map.update({r:'Other'       for r in other})

broad_order = ['Visual cortex','Thalamus','Hippocampus','Other']
permute_by_region = []
for broad in broad_order:
    for i,label in enumerate(unit_labels_permuted):
        if broad_map[label] == broad:
            permute_by_region.append(i)

def find_runs(labels):
    runs = []
    start = 0
    prev  = labels[0]
    for i,lab in enumerate(labels[1:],1):
        if lab != prev:
            runs.append((prev,start,i-1))
            prev, start = lab, i
    runs.append((prev,start,len(labels)-1))
    return runs

# Circle Layout for Clusters
def get_clustered_circle_positions(mem,
                                   big_radius    = 10.0,
                                   ring_radius   = 2.0,
                                   perim_thresh  = 8,
                                   disk_radius   = 3.0):
    clusters = defaultdict(list)
    for idx, c in enumerate(mem):
        clusters[c].append(idx)

    unique_cs = sorted(clusters.keys())
    K = len(unique_cs)
    pos = {}

    golden_angle = np.pi * (3 - np.sqrt(5))

    for ci, c in enumerate(unique_cs):
        θc = 2 * np.pi * ci / K
        cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)

        nodes = clusters[c]
        n = len(nodes)

        if n <= perim_thresh:
            for j, node in enumerate(nodes):
                θ = 2*np.pi * j / n
                x = cx + ring_radius * np.cos(θ)
                y = cy + ring_radius * np.sin(θ)
                pos[node] = (x, y)
        else:
            for j, node in enumerate(nodes):
                r = disk_radius * np.sqrt((j + 0.5)/n)
                θ = j * golden_angle
                x, y = cx + r * np.cos(θ), cy + r * np.sin(θ)
                pos[node] = (x, y)

    return pos

# Plotting Functions
def plot_graph_with_edges(G, stim, pos):
    fig, ax = plt.subplots(figsize=(10,10))
    
    # Nodes: region fill color
    for i in G.nodes():
        reg = unit_labels_permuted[i]
        fc  = color_map[reg]
        x,y = pos[i]
        ax.scatter(
            x, y,
            s=150,
            facecolor=fc,
            edgecolor='k',
            linewidth=0.5,
            zorder=3
        )

    # Edges: draw arrows
    for u, v in G.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        ax.annotate(
            '', xy=(x2, y2), xytext=(x1, y1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.5, alpha=0.6),
            size=10
        )

    ax.set_title(f"{stim}: Directed Graph Before Clustering", fontsize=14, pad=20)
    ax.axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(plot_save_directory, f"Graph_{stim}_WithEdges.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_graph_clusters(mem, stim):
    N = len(units)
    pos = get_clustered_circle_positions(mem,
                                   big_radius   = 8.0,
                                   ring_radius = 1.0,
                                   perim_thresh = 8,
                                   disk_radius = 3.0)
    fig, ax = plt.subplots(figsize=(8,8))
    for i in range(N):
        reg = unit_labels_permuted[i]
        fc  = color_map[reg]
        ec  = cmap10(mem[i] % 10)
        x,y = pos[i]
        ax.scatter(
            x, y,
            s=200,
            facecolor=fc,
            edgecolor=ec,
            linewidth=2,
            zorder=2
        )
    ax.set_title(f"{stim}: Nodes & SCC clusters", pad=20, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(plot_save_directory, f"Graph_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_adj_clusters(avgS, mem, stim):
    order    = permute_by_region
    Aord     = avgS[np.ix_(order, order)]
    regs_ord = [unit_labels_permuted[i] for i in order]
    region_runs = find_runs(regs_ord)
    mem_ord  = [mem[i] for i in order]

    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(Aord, aspect='auto', cmap='viridis', vmin=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('Avg |Weight|')

    for _,s,e in region_runs:
        ax.axhline(e+0.5, color='white', lw=0.8)
        ax.axvline(e+0.5, color='white', lw=0.8)

    for col, c in enumerate(mem_ord):
        ax.bar(col, 0.02, bottom=1.0, width=1.0,
               color=cmap10(c % 10),
               transform=ax.get_xaxis_transform(),
               clip_on=False)
    for row, c in enumerate(mem_ord):
        ax.barh(row, 0.02, left=-0.02, height=1.0,
                color=cmap10(c % 10),
                transform=ax.get_yaxis_transform(),
                clip_on=False)

    for lbl in ax.get_yticklabels():
        lbl.set_ha('right')
    ax.tick_params(axis='y', pad=10)
    plt.subplots_adjust(left=0.25, bottom=0.2)

    rticks  = [(s+e)/2 for (_,s,e) in region_runs]
    rlabels = [r for (r,s,e) in region_runs]
    ax.set_xticks(rticks)
    ax.set_xticklabels(rlabels, rotation=90, fontsize=8)
    ax.set_yticks(rticks)
    ax.set_yticklabels(rlabels, fontsize=8)

    ax.set_title(f"{stim}: adjacency by region & SCC", pad=30, fontsize=12)
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)

    plt.tight_layout(rect=[0,0.05,1,1])
    fig.savefig(os.path.join(plot_save_directory,
                             f"AdjMatrix_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

# MAIN: Load your stim_edge_data first
#stim_edge_data = {}  # Load or define your (avg_strength, stability, n_trials) here

for stim, (avgS, stabS, nT) in stim_edge_data.items():
    print(f">>> {stim}  trials={nT}")

    # Build directed graph
    G = nx.DiGraph()
    N = avgS.shape[0]
    for i in range(N):
        for j in range(N):
            if stabS[i, j] >= min_stability and avgS[i, j] != 0:
                G.add_edge(i, j)

    # Get positions for graph visualization
    dummy_mem = np.zeros(N, dtype=int)  # dummy mem for layout
    pos = get_clustered_circle_positions(dummy_mem)

    # 🔥 Plot Graph with Edges
    plot_graph_with_edges(G, stim, pos)

    # SCC Clustering
    sccs = list(nx.strongly_connected_components(G))
    mem = np.zeros(N, dtype=int) - 1
    for cluster_id, scc in enumerate(sccs):
        for node in scc:
            mem[node] = cluster_id

    # Regular plotting
    plot_graph_clusters(mem, stim)
    plot_adj_clusters(avgS, mem, stim)

print("✅ SCC clustering and plotting complete.")


In [ ]:
import os
print(os.getcwd())

In [ ]:
# --- IMPORTS ---
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import networkx as nx
import pandas as pd
import seaborn as sns
from collections import defaultdict
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.lines import Line2D

# --- CONFIGURATION ---
save_directory = NP_SAVE
plot_save_directory = NP_FIGS
os.makedirs(plot_save_directory, exist_ok=True)

sess_id = 791319847
stim_label_list = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha = 0.05
bin_size = 0.01
niter = 50
min_stability = 0.8

# --- COLOR GROUPS ---
visual_cortex = ['VISp', 'VISl', 'VISrl', 'VISal', 'VISpm', 'VISam']
hippocampus = ['CA1', 'CA2', 'CA3', 'DG', 'SUB']
thalamus = ['LGv', 'LP']
other = ['POL']

def get_shaded_colors(region_list, base_cmap, start=0.3, end=0.9):
    cmap = cm.get_cmap(base_cmap)
    n = len(region_list)
    shades = np.linspace(start, end, n)
    return {region: cmap(shades[i]) for i, region in enumerate(region_list)}

color_group_map = {}
color_group_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_group_map.update(get_shaded_colors(hippocampus, 'Reds'))
color_group_map.update(get_shaded_colors(thalamus, 'Greens'))
color_group_map.update(get_shaded_colors(other, 'Purples'))

# --- CUSTOM LAYOUT FUNCTION ---
def get_clustered_circle_positions_grouped(labels, min_spacing=6, base_cluster_radius=3.0):
    pos = {}
    region_indices = defaultdict(list)
    for i, label in enumerate(labels):
        region_indices[label].append(i)

    region_groups = [visual_cortex, hippocampus, thalamus, other]
    region_names = [r for group in region_groups for r in group if r in region_indices]
    num_regions = len(region_names)
    region_sizes = {label: len(region_indices[label]) for label in region_names}
    max_cluster_radius = base_cluster_radius * max(region_sizes.values()) ** 0.5
    circle_radius = min_spacing * num_regions + max_cluster_radius * 2
    theta_regions = np.linspace(0, 2 * np.pi, num_regions, endpoint=False)

    for k, (label, theta) in enumerate(zip(region_names, theta_regions)):
        indices = region_indices[label]
        n = len(indices)
        cluster_radius = base_cluster_radius * (n ** 0.5)
        cx = np.cos(theta) * circle_radius
        cy = np.sin(theta) * circle_radius
        theta_neurons = np.linspace(0, 2 * np.pi, n, endpoint=False)
        for j, idx in enumerate(indices):
            dx = np.cos(theta_neurons[j]) * cluster_radius
            dy = np.sin(theta_neurons[j]) * cluster_radius
            pos[idx] = (cx + dx, cy + dy)

    return pos

# --- LOAD DATA (First Pass: Average Strength & Stability Matrices) ---
stim_edge_data = {}
global_min_weight = float('inf')
global_max_weight = float('-inf')

for stim_label in stim_label_list:
    ls = 60 if stim_label != 'flashes' else 3
    edge_strength_raw = np.zeros((len(units), len(units)))
    edge_presence_raw = np.zeros((len(units), len(units)))
    unit_to_full_index = {unit_id: i for i, unit_id in enumerate(units)}
    n_trials = 0

    for idx in range(ls):
        file = f"{save_directory}/ID{sess_id}_{stim_label}_bin{bin_size}_alpha{alpha}_idx{idx}_niter{niter}.p"
        if not os.path.exists(file):
            continue

        with open(file, 'rb') as f:
            result = pkl.load(f)

        ce = result['ce']
        ce = ce - np.diag(np.diag(ce))
        trial_units = result.get('units', units_to_use_idx_[stim_label])

        try:
            idx_map = [unit_to_full_index[uid] for uid in trial_units]
        except KeyError:
            continue

        for i, idx_i in enumerate(idx_map):
            for j, idx_j in enumerate(idx_map):
                edge_strength_raw[idx_i, idx_j] += abs(ce[i, j])
                if ce[i, j] != 0:
                    edge_presence_raw[idx_i, idx_j] += 1

        n_trials += 1

    if n_trials == 0:
        continue

    avg_strength = edge_strength_raw / n_trials
    stability = edge_presence_raw / n_trials
    avg_strength[stability < min_stability] = 0
    stability[stability < min_stability] = 0

    weights = np.abs(avg_strength[avg_strength != 0])
    if weights.size > 0:
        global_min_weight = min(global_min_weight, weights.min())
        global_max_weight = max(global_max_weight, weights.max())

    stim_edge_data[stim_label] = (avg_strength, stability, n_trials)

# --- SECOND PASS: Plot Graph, SCC Clustering, and Adjacency Matrix ---
max_width_scale = 1.0
min_alpha = 0.3

for stim_label, (avg_strength, stability, n_trials) in stim_edge_data.items():
    print(f"\n--- Processing stimulus: {stim_label} ---")

    pos = get_clustered_circle_positions_grouped(unit_labels_permuted)
    # ADD THIS — Apply permutation first
    avg_strength_perm = avg_strength[np.ix_(permute, permute)]
    stability_perm = stability[np.ix_(permute, permute)]
    G = nx.DiGraph()
    for i in range(len(units)):
        label = unit_labels_permuted[i]
        G.add_node(i, area=label)

    for i in range(len(units)):
        for j in range(len(units)):
            #w = avg_strength[i, j]
            #s = stability[i, j]
            w = avg_strength_perm[i, j]
            s = stability_perm[i, j]
            if w != 0:
                G.add_edge(i, j, weight=w, stability=s)

    print(f" --> Built graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

    fig, ax = plt.subplots(figsize=(12, 10))
    node_colors = [color_group_map[G.nodes[n]['area']] for n in G.nodes]

    for (u, v, d) in G.edges(data=True):
        alpha_val = min_alpha + (1 - min_alpha) * d['stability']
        width_val = np.interp(np.abs(d['weight']), [global_min_weight, global_max_weight], [0.1, max_width_scale])
        nx.draw_networkx_edges(
            G, pos,
            edgelist=[(u, v)],
            width=width_val,
            alpha=alpha_val,
            edge_color='gray',
            arrows=True,
            ax=ax,
            connectionstyle='arc3,rad=0.2'
        )

    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=100, ax=ax)
    nx.draw_networkx_labels(G, pos, labels={i: str(i) for i in G.nodes}, font_size=6, ax=ax)

    # Legends
    subregion_labels = list(dict.fromkeys([unit_labels_permuted[i] for i in G.nodes]))
    legend_patches = [mpatches.Patch(color=color_group_map[label], label=label) for label in subregion_labels]
    ax.legend(
        handles=legend_patches,
        title='Brain Subregions',
        loc='lower center',
        bbox_to_anchor=(0.5, -0.32),
        ncol=4,
        fontsize=11,
        title_fontsize=12,
        frameon=False
    )

    # Edge Thickness Legend
    edge_legend_y_base = -0.42
    ax.annotate('Edge Thickness ~ |Weight|', xy=(0.5, edge_legend_y_base), xycoords='axes fraction', ha='center', fontsize=11)
    legend_weights = [global_min_weight, (global_min_weight + global_max_weight) / 2, global_max_weight]
    legend_labels = [f"{w:.2f}" for w in legend_weights]
    legend_widths = [np.interp(w, [global_min_weight, global_max_weight], [0.1, max_width_scale]) for w in legend_weights]

    for i, (w_val, w_lbl) in enumerate(zip(legend_widths, legend_labels)):
        x = 0.4 + i * 0.1
        y = edge_legend_y_base - 0.04
        ax.annotate(
            '', xy=(x + 0.02, y), xytext=(x, y),
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='->', lw=w_val * 1.0, color='black')
        )
        ax.text(x + 0.03, y, w_lbl, transform=ax.transAxes, va='center', fontsize=10)

    # Stability Colorbar
    cb_ax1 = fig.add_axes([0.77, 0.05, 0.2, 0.02])
    cb1 = plt.colorbar(plt.cm.ScalarMappable(cmap='Greys'), cax=cb_ax1, orientation='horizontal')
    cb1.set_label('Stability (Edge Transparency)', fontsize=11)
    cb1.ax.tick_params(labelsize=10)
    cb1.set_ticks([min_alpha, 1.0])
    cb1.set_ticklabels([f"{min_alpha:.2f}", "1.00"])

    fig.subplots_adjust(bottom=0.45, right=0.75)

    ax.set_title(f'Graph: {stim_label} - Avg Connectivity Across {n_trials} Trials', fontsize=13)
    ax.axis('off')

    basename = f"{plot_save_directory}Graph_ID{sess_id}_{stim_label}"
    fig.savefig(basename + ".pdf", dpi=600, bbox_inches='tight')
    fig.savefig(basename + ".png", dpi=600, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved graph: {basename}.pdf and .png")

    # Now SCC clustering
    sccs = list(nx.strongly_connected_components(G))
    mem = np.zeros(len(units), dtype=int) #- 1
    for cluster_id, scc in enumerate(sccs):
        for node in scc:
            mem[node] = cluster_id

    # Plot SCC Clusters
    fig, ax = plt.subplots(figsize=(10, 8))
    node_colors_cluster = [color_group_map[unit_labels_permuted[i]] for i in G.nodes]
    edge_colors_cluster = [f"C{mem[i] % 10}" for i in G.nodes]

    for i in range(len(units)):
        x, y = pos[i]
        ax.scatter(
            x, y,
            s=150,
            facecolor=node_colors_cluster[i],
            edgecolor=edge_colors_cluster[i],
            linewidth=2,
            zorder=2
        )

    ax.set_title(f"{stim_label}: SCC Clusters", fontsize=14, pad=20)
    ax.axis('off')
    plt.tight_layout()
    fig.savefig(f"{plot_save_directory}Graph_SCC_{stim_label}.png", dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Saved SCC clustered plot: {plot_save_directory}Graph_SCC_{stim_label}.png")


In [ ]:
import os
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx
from collections import defaultdict

# --- CONFIGURATION & DATA LOADING (assume existing) ---
save_directory       = NP_SAVE
plot_save_directory  = NP_FIGS
os.makedirs(plot_save_directory, exist_ok=True)

sess_id              = 791319847
stim_label_list      = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha, bin_size, niter = 0.05, 0.01, 50
min_stability        = 0.8

# --- REGION COLORS ---
visual_cortex = ['VISp','VISl','VISrl','VISal','VISpm','VISam']
thalamus      = ['LGv','LP']
hippocampus   = ['CA1','CA2','CA3','DG','SUB']
other         = ['POL']

def get_shaded_colors(regions, cmap_name):
    cmap   = cm.get_cmap(cmap_name)
    shades = np.linspace(0.3,0.9,len(regions))
    return {r: cmap(shades[i]) for i,r in enumerate(regions)}

color_map = {}
color_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_map.update(get_shaded_colors(thalamus,      'Greens'))
color_map.update(get_shaded_colors(hippocampus,   'Reds'))
color_map.update(get_shaded_colors(other,         'Purples'))

cmap10 = plt.get_cmap('tab10')

# --- PERMUTE BY BROAD REGION ORDER ---
broad_map = {r:'Visual cortex' for r in visual_cortex}
broad_map.update({r:'Thalamus'    for r in thalamus})
broad_map.update({r:'Hippocampus' for r in hippocampus})
broad_map.update({r:'Other'       for r in other})

broad_order = ['Visual cortex','Thalamus','Hippocampus','Other']
permute_by_region = []
for broad in broad_order:
    for i,label in enumerate(unit_labels_permuted):
        if broad_map[label] == broad:
            permute_by_region.append(i)

# --- Helper: find contiguous runs ---
def find_runs(labels):
    runs = []
    start = 0
    prev  = labels[0]
    for i,lab in enumerate(labels[1:],1):
        if lab != prev:
            runs.append((prev,start,i-1))
            prev, start = lab, i
    runs.append((prev,start,len(labels)-1))
    return runs

# --- BUILD per-stimulus avg & stability matrices ---
stim_edge_data = {}
for stim in stim_label_list:
    ls       = 3 if stim == 'flashes' else 60
    ER       = np.zeros((len(units),len(units)))
    EP       = np.zeros_like(ER)
    u2i      = {u:i for i,u in enumerate(units)}
    n_trials = 0
    for tr in range(ls):
        fn = f"{save_directory}/ID{sess_id}_{stim}_bin{bin_size}_alpha{alpha}_idx{tr}_niter{niter}.p"
        if not os.path.exists(fn):
            continue
        R = pkl.load(open(fn,'rb'))
        ce = R['ce'] - np.diag(np.diag(R['ce']))
        trial_idxs = R.get('units', units_to_use_idx_[stim])
        try:
            locs = [u2i[u] for u in trial_idxs]
        except KeyError:
            continue
        for i,i0 in enumerate(locs):
            for j,j0 in enumerate(locs):
                if ce[i,j] != 0:
                    EP[i0,j0] += 1
                    ER[i0,j0] += abs(ce[i,j])
        n_trials += 1
    if n_trials == 0:
        continue
    avg = ER / n_trials
    stab= EP / n_trials
    avg[stab < min_stability] = 0
    stab[stab < min_stability] = 0
    stim_edge_data[stim] = (avg, stab, n_trials)

# --- Plot functions for SCC ---
def plot_graph_scc(mem, stim):
    N = len(units)

    # NEW: Get regular and singleton clusters before position
    clusters = defaultdict(list)
    for idx, c in enumerate(mem):
        clusters[c].append(idx)

    singletons = [idxs[0] for key, idxs in clusters.items() if len(idxs) == 1 and key != -1]
    unassigned_nodes = [idxs[0] for key, idxs in clusters.items() if key == -1]
    regular_clusters = [idxs for key, idxs in clusters.items() if len(idxs) > 1]
    
    #singletons = [idxs[0] for idxs in clusters.values() if len(idxs) == 1]
    #regular_clusters = [idxs for idxs in clusters.values() if len(idxs) > 1]

    pos = get_clustered_circle_positions(mem, singletons, regular_clusters,
                                   big_radius=8.0,
                                   ring_radius=1.0,
                                   perim_thresh=8,
                                   disk_radius=3.0)

    fig, ax = plt.subplots(figsize=(8, 8))
    for i in range(N):
        reg = unit_labels_permuted[i]
        fc = color_map[reg]
        x, y = pos[i]
        ax.scatter(
            x, y,
            s=200,
            facecolor=fc,
            edgecolor='k',  # Thin black outline
            linewidth=0.5,
            zorder=2
        )

#    Draw dotted circles around only regular clusters
    for nodes in regular_clusters:
        pts = np.array([pos[i] for i in nodes])
        center = pts.mean(axis=0)
        dists = np.linalg.norm(pts - center, axis=1)
        radius = dists.max() + 1.0
        circle = plt.Circle(center, radius,
                            color='gray',  # Fixed gray
                            linestyle='dotted',
                            linewidth=2.0,
                            fill=False,
                            zorder=1)
        ax.add_patch(circle)

    # (NO circle for singletons — just placed together)

    ax.set_title(f"{stim}: Nodes & SCC clusters", pad=20, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(plot_save_directory, f"Graph_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)
# def plot_graph_scc(mem, stim):
#     N = len(units)

#     # 1) Build a map: cluster_id → list of node indices
#     clusters = defaultdict(list)
#     for idx, c in enumerate(mem):
#         clusters[c].append(idx)

#     # 2) Identify three categories:
#     #    (a) regular_clusters: cluster size > 1 and cluster_id != -1
#     #    (b) singletons: exactly one node, but cluster_id != -1
#     #    (c) unassigned: cluster_id == -1
#     regular_clusters = [
#         idxs
#         for key, idxs in clusters.items()
#         if len(idxs) > 1 and key != -1
#     ]
#     singletons = [
#         idxs[0]
#         for key, idxs in clusters.items()
#         if len(idxs) == 1 and key != -1
#     ]
#     # (We do not need to explicitly list unassigned here—just skip them when drawing circles.)

#     # 3) Get every node’s (x,y) using your original single‐argument layout.
#     #    (This layout already placed unassigned nodes somewhere, so no KeyError.)
#     pos = get_clustered_circle_positions(mem)

#     fig, ax = plt.subplots(figsize=(8, 8))

#     # 4) Plot all nodes at their pos[i] (including unassigned)
#     for i in range(N):
#         reg = unit_labels_permuted[i]
#         fc = color_map[reg]
#         x, y = pos[i]
#         ax.scatter(
#             x, y,
#             s=200,
#             facecolor=fc,
#             edgecolor='k',  # thin black outline
#             linewidth=0.5,
#             zorder=2
#         )

#     # 5) Draw a large dotted circle around each multi‐node SCC
#     for nodes in regular_clusters:
#         pts = np.array([pos[i] for i in nodes])
#         center = pts.mean(axis=0)
#         dists = np.linalg.norm(pts - center, axis=1)
#         radius = dists.max() + 1.0
#         circle = plt.Circle(
#             center, radius,
#             color='gray',
#             linestyle='dotted',
#             linewidth=2.0,
#             fill=False,
#             zorder=1
#         )
#         ax.add_patch(circle)

#     # 6) Draw a very tight dotted circle around each true singleton
#     singleton_radius = 0.3
#     for node in singletons:
#         x, y = pos[node]
#         tiny_circle = plt.Circle(
#             (x, y),
#             singleton_radius,
#             color='gray',
#             linestyle='dotted',
#             linewidth=1.5,
#             fill=False,
#             zorder=1
#         )
#         ax.add_patch(tiny_circle)
#     # (We do NOT draw any circle for nodes with mem == -1)

#     ax.set_title(f"{stim}: Nodes & SCC clusters", pad=20, fontsize=14)
#     ax.axis('off')
#     fig.tight_layout()
#     fig.savefig(
#         os.path.join(plot_save_directory, f"Graph_{stim}_SCC.png"),
#         dpi=300, bbox_inches='tight'
#     )
#     plt.close(fig)


# def plot_graph_scc(mem, stim):
#     N = len(units)

#     # NEW: Get regular and singleton clusters before position
#     clusters = defaultdict(list)
#     for idx, c in enumerate(mem):
#         clusters[c].append(idx)

#     singletons = [idxs[0] for key, idxs in clusters.items() if len(idxs) == 1 and key != -1]
#     unassigned_nodes = [idxs[0] for key, idxs in clusters.items() if key == -1]
#     regular_clusters = [idxs for key, idxs in clusters.items() if len(idxs) > 1 and key != -1]
    
#     #singletons = [idxs[0] for idxs in clusters.values() if len(idxs) == 1]
#     #regular_clusters = [idxs for idxs in clusters.values() if len(idxs) > 1]

#     pos = get_clustered_circle_positions(mem, singletons, regular_clusters,
#                                    big_radius=8.0,
#                                    ring_radius=1.0,
#                                    perim_thresh=8,
#                                    disk_radius=3.0)

#     fig, ax = plt.subplots(figsize=(8, 8))
#     for i in range(N):
#         reg = unit_labels_permuted[i]
#         fc = color_map[reg]
#         x, y = pos[i]
#         ax.scatter(
#             x, y,
#             s=200,
#             facecolor=fc,
#             edgecolor='k',  # Thin black outline
#             linewidth=0.5,
#             zorder=2
#         )

# #    Draw dotted circles around only regular clusters
#     for nodes in regular_clusters:
#         pts = np.array([pos[i] for i in nodes])
#         center = pts.mean(axis=0)
#         dists = np.linalg.norm(pts - center, axis=1)
#         radius = dists.max() + 1.0
#         circle = plt.Circle(center, radius,
#                             color='gray',  # Fixed gray
#                             linestyle='dotted',
#                             linewidth=2.0,
#                             fill=False,
#                             zorder=1)
#         ax.add_patch(circle)

#     # (NO circle for singletons — just placed together)

#     ax.set_title(f"{stim}: Nodes & SCC clusters", pad=20, fontsize=14)
#     ax.axis('off')
#     fig.tight_layout()
#     fig.savefig(os.path.join(plot_save_directory, f"Graph_{stim}_SCC.png"),
#                 dpi=300, bbox_inches='tight')
#     plt.close(fig)

def get_clustered_circle_positions(mem, singletons, regular_clusters,
                                   big_radius=6.0,     # <-- shrink cluster spread
                                   ring_radius=1.0,
                                   perim_thresh=8,
                                   disk_radius=3.0,
                                   singleton_disk_radius=2.5):  # <-- dense singleton cluster
    pos = {}
    golden_angle = np.pi * (3 - np.sqrt(5))

    # Position regular clusters first
    K = len(regular_clusters) + 1  # +1 slot for singletons
    for ci, nodes in enumerate(regular_clusters):
        θc = 2 * np.pi * ci / K
        cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)
        n = len(nodes)

        if n <= perim_thresh:
            for j, node in enumerate(nodes):
                θ = 2 * np.pi * j / n
                x = cx + ring_radius * np.cos(θ)
                y = cy + ring_radius * np.sin(θ)
                pos[node] = (x, y)
        else:
            for j, node in enumerate(nodes):
                r = disk_radius * np.sqrt((j + 0.5) / n)
                θ = j * golden_angle
                x = cx + r * np.cos(θ)
                y = cy + r * np.sin(θ)
                pos[node] = (x, y)

    # Position singleton nodes *inside* a disk
    if singletons:
        θc = 2 * np.pi * (K - 1) / K
        cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)

        n_singletons = len(singletons)
        for j, node in enumerate(singletons):
            r = singleton_disk_radius * np.sqrt((j + 0.5) / n_singletons)
            θ = j * golden_angle
            x = cx + r * np.cos(θ)
            y = cy + r * np.sin(θ)
            pos[node] = (x, y)

    return pos




def plot_adj_scc(avgS, mem, stim):
    order    = permute_by_region
    Aord     = avgS[np.ix_(order, order)]
    regs_ord = [unit_labels_permuted[i] for i in order]
    region_runs = find_runs(regs_ord)
    mem_ord  = [mem[i] for i in order]

    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(Aord, aspect='auto', cmap='viridis', vmin=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('Avg |Weight|')

    # region boundaries
    for _,s,e in region_runs:
        ax.axhline(e+0.5, color='white', lw=0.8)
        ax.axvline(e+0.5, color='white', lw=0.8)

    # top stripe (SCC cluster color bar)
    for col, c in enumerate(mem_ord):
        ax.bar(col, 0.02, bottom=1.0, width=1.0,
               color=cmap10(c % 10),
               transform=ax.get_xaxis_transform(),
               clip_on=False)
    # left stripe
    for row, c in enumerate(mem_ord):
        ax.barh(row, 0.02, left=-0.02, height=1.0,
                color=cmap10(c % 10),
                transform=ax.get_yaxis_transform(),
                clip_on=False)

    # adjust y-ticks
    for lbl in ax.get_yticklabels():
        lbl.set_ha('right')
    ax.tick_params(axis='y', pad=10)
    plt.subplots_adjust(left=0.25, bottom=0.2)

    # region tick labels
    rticks  = [(s+e)/2 for (_,s,e) in region_runs]
    rlabels = [r for (r,s,e) in region_runs]
    ax.set_xticks(rticks)
    ax.set_xticklabels(rlabels, rotation=90, fontsize=8)
    ax.set_yticks(rticks)
    ax.set_yticklabels(rlabels, fontsize=8)

    ax.set_title(f"{stim}: adjacency by region & SCC", pad=30, fontsize=12)
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)

    plt.tight_layout(rect=[0,0.05,1,1])
    fig.savefig(os.path.join(plot_save_directory,
                             f"AdjMatrix_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)

# --- MAIN: Run SCC clustering and plot ---
# def get_clustered_circle_positions(mem,
#                                    big_radius    = 10.0,
#                                    ring_radius   = 2.0,
#                                    perim_thresh  = 8,
#                                    disk_radius   = 3.0):
#     clusters = defaultdict(list)
#     for idx, c in enumerate(mem):
#         clusters[c].append(idx)

#     unique_cs = sorted(clusters.keys())
#     K = len(unique_cs)
#     pos = {}

#     golden_angle = np.pi * (3 - np.sqrt(5))  # ~2.39996

#     for ci, c in enumerate(unique_cs):
#         θc = 2 * np.pi * ci / K
#         cx, cy = big_radius*np.cos(θc), big_radius*np.sin(θc)

#         nodes = clusters[c]
#         n     = len(nodes)

#         if n <= perim_thresh:
#             for j, node in enumerate(nodes):
#                 θ = 2*np.pi * j / n
#                 x = cx + ring_radius * np.cos(θ)
#                 y = cy + ring_radius * np.sin(θ)
#                 pos[node] = (x, y)
#         else:
#             for j, node in enumerate(nodes):
#                 r     = disk_radius * np.sqrt((j + 0.5)/n)
#                 θ     = j * golden_angle
#                 x, y  = cx + r*np.cos(θ), cy + r*np.sin(θ)
#                 pos[node] = (x, y)
#     return pos

# def get_clustered_circle_positions(mem,
#                                    big_radius    = 10.0,
#                                    ring_radius   = 2.0,
#                                    perim_thresh  = 8,
#                                    disk_radius   = 3.0):
#     clusters = defaultdict(list)
#     for idx, c in enumerate(mem):
#         clusters[c].append(idx)

#     unique_cs = sorted(clusters.keys())
#     # Separate singleton clusters
#     normal_clusters = [c for c in unique_cs if len(clusters[c]) > 1]
#     singleton_nodes = [idx for c in unique_cs if len(clusters[c]) == 1 for idx in clusters[c]]

#     K = len(normal_clusters)
#     pos = {}

#     golden_angle = np.pi * (3 - np.sqrt(5))  # ~2.39996

#     # Position normal clusters on large circle
#     for ci, c in enumerate(normal_clusters):
#         θc = 2 * np.pi * ci / max(K, 1)
#         cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)

#         nodes = clusters[c]
#         n     = len(nodes)

#         if n <= perim_thresh:
#             for j, node in enumerate(nodes):
#                 θ = 2 * np.pi * j / n
#                 x = cx + ring_radius * np.cos(θ)
#                 y = cy + ring_radius * np.sin(θ)
#                 pos[node] = (x, y)
#         else:
#             for j, node in enumerate(nodes):
#                 r = disk_radius * np.sqrt((j + 0.5) / n)
#                 θ = j * golden_angle
#                 x, y = cx + r * np.cos(θ), cy + r * np.sin(θ)
#                 pos[node] = (x, y)

#     # Position all singleton nodes together
#     if singleton_nodes:
#         cx, cy = 0, 0  # center singleton group at origin
#         n_singletons = len(singleton_nodes)
#         singleton_ring_radius = 3.0  # tighter circle

#         for j, node in enumerate(singleton_nodes):
#             θ = 2 * np.pi * j / n_singletons
#             x = cx + singleton_ring_radius * np.cos(θ)
#             y = cy + singleton_ring_radius * np.sin(θ)
#             pos[node] = (x, y)

#     return pos



for stim, (avgS, stabS, nT) in stim_edge_data.items():
    print(f">>> {stim}  trials={nT}")
    avgS_perm = avgS[np.ix_(permute, permute)]
    # G = nx.DiGraph()
    # for i in range(len(units)):
    #     for j in range(len(units)):
    #         if avgS_perm[i, j] != 0:
    #             G.add_edge(i, j)
    G = nx.DiGraph()
    
    # Normalize strengths first
    strengths = avgS_perm
    max_strength = np.max(strengths)
    if max_strength == 0:
        max_strength = 1  # Avoid division by zero
    
    normalized_strengths = strengths / max_strength
    
    # Set your threshold here (tune as needed)
    combined_threshold = 0.3  # Example: 0.4
    
    for i in range(len(units)):
        for j in range(len(units)):
            stability_ij = stabS[np.ix_(permute, permute)][i, j]
            strength_norm_ij = normalized_strengths[i, j]
            combined_score = stability_ij * strength_norm_ij
    
            if combined_score >= combined_threshold:
                G.add_edge(i, j)

    sccs = list(nx.strongly_connected_components(G))

    # --- FILTER: Only keep clusters with size >= th
    th = 0
    sccs_filtered = [scc for scc in sccs if len(scc) >= th]

    # --- New membership assignment ---
    mem = np.zeros(len(units), dtype=int) - 1  # default -1 (unassigned)
    for cluster_id, scc in enumerate(sccs_filtered):
        for node in scc:
            mem[node] = cluster_id
            # Print connections inside each cluster
    for cluster_id, scc in enumerate(sccs_filtered):
        nodes_in_cluster = list(scc)
        print(f"\nCluster {cluster_id+1}: {len(nodes_in_cluster)} nodes")
        for i in nodes_in_cluster:
            for j in nodes_in_cluster:
                if G.has_edge(i, j):
                    src_label = unit_labels_permuted[i]
                    tgt_label = unit_labels_permuted[j]
                    print(f"  {i} ({src_label}) → {j} ({tgt_label})")


    plot_graph_scc(mem, stim)
    plot_adj_scc(avgS, mem, stim)


print("✅ SCC-only final plots complete.")

In [ ]:
import os
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import networkx as nx
from collections import defaultdict

# --- CONFIGURATION & DATA LOADING (assume existing) ---
save_directory       = NP_SAVE
plot_save_directory  = NP_FIGS
os.makedirs(plot_save_directory, exist_ok=True)

sess_id              = 791319847
stim_label_list      = ['natural_scenes', 'static_gratings', 'gabors', 'flashes']
alpha, bin_size, niter = 0.05, 0.01, 50
min_stability        = 0.8

# --- REGION COLORS ---
visual_cortex = ['VISp','VISl','VISrl','VISal','VISpm','VISam']
thalamus      = ['LGv','LP']
hippocampus   = ['CA1','CA2','CA3','DG','SUB']
other         = ['POL']

def get_shaded_colors(regions, cmap_name):
    cmap   = cm.get_cmap(cmap_name)
    shades = np.linspace(0.3,0.9,len(regions))
    return {r: cmap(shades[i]) for i,r in enumerate(regions)}

color_map = {}
color_map.update(get_shaded_colors(visual_cortex, 'Blues'))
color_map.update(get_shaded_colors(thalamus,      'Greens'))
color_map.update(get_shaded_colors(hippocampus,   'Reds'))
color_map.update(get_shaded_colors(other,         'Purples'))

cmap10 = plt.get_cmap('tab10')

# --- PERMUTE BY BROAD REGION ORDER ---
broad_map = {r:'Visual cortex' for r in visual_cortex}
broad_map.update({r:'Thalamus'    for r in thalamus})
broad_map.update({r:'Hippocampus' for r in hippocampus})
broad_map.update({r:'Other'       for r in other})

broad_order = ['Visual cortex','Thalamus','Hippocampus','Other']
permute_by_region = []
for broad in broad_order:
    for i,label in enumerate(unit_labels_permuted):
        if broad_map[label] == broad:
            permute_by_region.append(i)

# --- Helper: find contiguous runs ---
def find_runs(labels):
    runs = []
    start = 0
    prev  = labels[0]
    for i,lab in enumerate(labels[1:],1):
        if lab != prev:
            runs.append((prev,start,i-1))
            prev, start = lab, i
    runs.append((prev,start,len(labels)-1))
    return runs

# --- BUILD per‐stimulus avg & stability matrices (unchanged) ---
stim_edge_data = {}
for stim in stim_label_list:
    ls       = 3 if stim == 'flashes' else 60
    ER       = np.zeros((len(units),len(units)))
    EP       = np.zeros_like(ER)
    u2i      = {u:i for i,u in enumerate(units)}
    n_trials = 0
    for tr in range(ls):
        fn = f"{save_directory}/ID{sess_id}_{stim}_bin{bin_size}_alpha{alpha}_idx{tr}_niter{niter}.p"
        if not os.path.exists(fn):
            continue
        R = pkl.load(open(fn,'rb'))
        ce = R['ce'] - np.diag(np.diag(R['ce']))
        trial_idxs = R.get('units', units_to_use_idx_[stim])
        try:
            locs = [u2i[u] for u in trial_idxs]
        except KeyError:
            continue
        for i,i0 in enumerate(locs):
            for j,j0 in enumerate(locs):
                if ce[i,j] != 0:
                    EP[i0,j0] += 1
                    ER[i0,j0] += abs(ce[i,j])
        n_trials += 1
    if n_trials == 0:
        continue
    avg = ER / n_trials
    stab= EP / n_trials
    avg[stab < min_stability] = 0
    stab[stab < min_stability] = 0
    stim_edge_data[stim] = (avg, stab, n_trials)

# =============================================================================
# === 1) Single‐argument “fallback” layout (exactly as before) ================

def get_clustered_circle_positions_simple(mem,
                                          big_radius    = 10.0,
                                          ring_radius   = 2.0,
                                          perim_thresh  = 8,
                                          disk_radius   = 3.0):
    """
    This is the original single‐argument version.  It returns a position for every node
    (including unassigned), based on mem only.  We'll call it “_simple” and use it
    as a fallback for any node missing in the 3‐arg layout.
    """
    clusters = defaultdict(list)
    for idx, c in enumerate(mem):
        clusters[c].append(idx)

    unique_cs = sorted(clusters.keys())
    K = len(unique_cs)
    pos = {}

    golden_angle = np.pi * (3 - np.sqrt(5))  # ~2.39996

    for ci, c in enumerate(unique_cs):
        θc = 2 * np.pi * ci / K
        cx, cy = big_radius*np.cos(θc), big_radius*np.sin(θc)

        nodes = clusters[c]
        n     = len(nodes)

        if n <= perim_thresh:
            for j, node in enumerate(nodes):
                θ = 2*np.pi * j / n
                x = cx + ring_radius * np.cos(θ)
                y = cy + ring_radius * np.sin(θ)
                pos[node] = (x, y)
        else:
            for j, node in enumerate(nodes):
                r     = disk_radius * np.sqrt((j + 0.5)/n)
                θ     = j * golden_angle
                x, y  = cx + r*np.cos(θ), cy + r*np.sin(θ)
                pos[node] = (x, y)

    return pos


# =============================================================================
# === 2) Three‐argument “custom” layout (keeps multi‐node SCCs and singletons) ==

def get_clustered_circle_positions(mem, singletons, regular_clusters,
                                   big_radius=6.0,     # for Gabors you can tweak this if needed
                                   ring_radius=1.0,
                                   perim_thresh=8,
                                   disk_radius=3.0,
                                   singleton_disk_radius=2.5):
    """
    3‐arg version:
     • regular_clusters = [ [i1,i2,…], [j1,j2,…], … ] 
       (only cluster_id != -1 and len>1)
     • singletons = [ i1, i2, … ]   (only cluster_id != -1 and len==1)
     The “unassigned” nodes (mem==-1) are simply not included here.
     We position each multi‐node SCC on the big circle, and each
     singleton inside its slot.  But after we compute this `pos`,
     we’ll fill in any missing indices from the fallback above.
    """
    pos = {}
    golden_angle = np.pi * (3 - np.sqrt(5))

    # Position multi‐node SCCs (regular_clusters) first
    K = len(regular_clusters) + 1  # +1 reserved “slot” for singletons
    for ci, nodes in enumerate(regular_clusters):
        θc = 2 * np.pi * ci / K
        cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)
        n = len(nodes)

        if n <= perim_thresh:
            for j, node in enumerate(nodes):
                θ = 2 * np.pi * j / n
                x = cx + ring_radius * np.cos(θ)
                y = cy + ring_radius * np.sin(θ)
                pos[node] = (x, y)
        else:
            for j, node in enumerate(nodes):
                r = disk_radius * np.sqrt((j + 0.5) / n)
                θ = j * golden_angle
                x = cx + r * np.cos(θ)
                y = cy + r * np.sin(θ)
                pos[node] = (x, y)

    # Now put singletons (cluster_id != -1, size==1) in the last slot
    if singletons:
        θc = 2 * np.pi * (K - 1) / K
        cx, cy = big_radius * np.cos(θc), big_radius * np.sin(θc)

        n_singletons = len(singletons)
        for j, node in enumerate(singletons):
            r = singleton_disk_radius * np.sqrt((j + 0.5) / n_singletons)
            θ = j * golden_angle
            x = cx + r * np.cos(θ)
            y = cy + r * np.sin(θ)
            pos[node] = (x, y)

    return pos


# =============================================================================
# === 3) plot_graph_scc with “no circle around unassigned” and “fallback pos” ===

def plot_graph_scc(mem, stim):
    N = len(units)

    # 3.1) Build cluster_id → [node indices]
    clusters = defaultdict(list)
    for idx, c in enumerate(mem):
        clusters[c].append(idx)

    # 3.2) Identify:
    #      regular_clusters = cluster_id != -1 AND size > 1
    #      singletons       = cluster_id != -1 AND size == 1
    #      unassigned       = cluster_id == -1  (we will skip them when drawing circles)
    regular_clusters = [
        idxs
        for key, idxs in clusters.items()
        if (key != -1 and len(idxs) > 1)
    ]
    singletons = [
        idxs[0]
        for key, idxs in clusters.items()
        if (key != -1 and len(idxs) == 1)
    ]
    # (no need to explicitly collect unassigned here)

    # 3.3) First compute “3‐arg” layout for regular + singletons
    pos = get_clustered_circle_positions(mem, singletons, regular_clusters,
                                         big_radius=8.0,
                                         ring_radius=1.0,
                                         perim_thresh=8,
                                         disk_radius=3.0)

    # 3.4) Now fill in any index i that is missing from pos → fallback to single‐arg layout
    pos_all = get_clustered_circle_positions_simple(mem)
    for i in range(N):
        if i not in pos:
            pos[i] = pos_all[i]

    fig, ax = plt.subplots(figsize=(8, 8))
    # > Plot every node (including unassigned) at pos[i]
    for i in range(N):
        reg = unit_labels_permuted[i]
        fc  = color_map[reg]
        x, y = pos[i]
        ax.scatter(
            x, y,
            s=200,
            facecolor=fc,
            edgecolor='k',
            linewidth=0.5,
            zorder=2
        )

    # 3.5) Draw dotted circles around each multi‐node SCC (regular_clusters)
    for nodes in regular_clusters:
        pts = np.array([pos[i] for i in nodes])
        center = pts.mean(axis=0)
        dists = np.linalg.norm(pts - center, axis=1)
        radius = dists.max() + 1.0
        circle = plt.Circle(
            center, radius,
            color='gray',
            linestyle='dotted',
            linewidth=2.0,
            fill=False,
            zorder=1
        )
        ax.add_patch(circle)

    # 3.6) Draw very tight dotted circles around each true singleton
    singleton_radius = 0.25
    for node in singletons:
        x, y = pos[node]
        tiny_circle = plt.Circle(
            (x, y),
            singleton_radius,
            color='gray',
            linestyle='dotted',
            linewidth=1.5,
            fill=False,
            zorder=1
        )
        ax.add_patch(tiny_circle)

    #  (We do NOT draw any circle for mem == -1)
    ax.set_title(f"{stim}: Nodes & SCC clusters", pad=20, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(os.path.join(plot_save_directory, f"Graph_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)


def plot_adj_scc(avgS, mem, stim):
    order    = permute_by_region
    Aord     = avgS[np.ix_(order, order)]
    regs_ord = [unit_labels_permuted[i] for i in order]
    region_runs = find_runs(regs_ord)
    mem_ord  = [mem[i] for i in order]

    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(Aord, aspect='auto', cmap='viridis', vmin=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('Avg |Weight|')

    # region boundaries
    for _,s,e in region_runs:
        ax.axhline(e+0.5, color='white', lw=0.8)
        ax.axvline(e+0.5, color='white', lw=0.8)

    # top stripe (SCC cluster color bar)
    for col, c in enumerate(mem_ord):
        ax.bar(col, 0.02, bottom=1.0, width=1.0,
               color=cmap10(c % 10),
               transform=ax.get_xaxis_transform(),
               clip_on=False)
    # left stripe
    for row, c in enumerate(mem_ord):
        ax.barh(row, 0.02, left=-0.02, height=1.0,
                color=cmap10(c % 10),
                transform=ax.get_yaxis_transform(),
                clip_on=False)

    # adjust y‐ticks
    for lbl in ax.get_yticklabels():
        lbl.set_ha('right')
    ax.tick_params(axis='y', pad=10)
    plt.subplots_adjust(left=0.25, bottom=0.2)

    # region tick labels
    rticks  = [(s+e)/2 for (_,s,e) in region_runs]
    rlabels = [r for (r,s,e) in region_runs]
    ax.set_xticks(rticks)
    ax.set_xticklabels(rlabels, rotation=90, fontsize=8)
    ax.set_yticks(rticks)
    ax.set_yticklabels(rlabels, fontsize=8)

    ax.set_title(f"{stim}: adjacency by region & SCC", pad=30, fontsize=12)
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)

    plt.tight_layout(rect=[0,0.05,1,1])
    fig.savefig(os.path.join(plot_save_directory,
                             f"AdjMatrix_{stim}_SCC.png"),
                dpi=300, bbox_inches='tight')
    plt.close(fig)


# --- MAIN: Run SCC clustering and plot =============================

for stim, (avgS, stabS, nT) in stim_edge_data.items():
    print(f">>> {stim}  trials={nT}")
    avgS_perm = avgS[np.ix_(permute, permute)]
    G = nx.DiGraph()

    # Normalize strengths first
    strengths = avgS_perm
    max_strength = np.max(strengths)
    if max_strength == 0:
        max_strength = 1  # avoid division by zero
    normalized_strengths = strengths / max_strength

    # threshold on (stability * normalized_strength)
    combined_threshold = 0.1
    for i in range(len(units)):
        for j in range(len(units)):
            stability_ij = stabS[np.ix_(permute, permute)][i, j]
            strength_norm_ij = normalized_strengths[i, j]
            if (stability_ij * strength_norm_ij) >= combined_threshold:
                G.add_edge(i, j)

    sccs = list(nx.strongly_connected_components(G))

    # keep all (we skip only if len(scc)<th, but here th=0)
    sccs_filtered = [scc for scc in sccs if len(scc) >= 0]

    # build mem (–1 means unassigned; otherwise cluster_id ≥ 0)
    mem = np.zeros(len(units), dtype=int) - 1
    for cluster_id, scc in enumerate(sccs_filtered):
        for node in scc:
            mem[node] = cluster_id

    # Print connections in each cluster
    for cluster_id, scc in enumerate(sccs_filtered):
        nodes_in_cluster = list(scc)
        print(f"\nCluster {cluster_id+1}: {len(nodes_in_cluster)} nodes")
        for i in nodes_in_cluster:
            for j in nodes_in_cluster:
                if G.has_edge(i, j):
                    src_label = unit_labels_permuted[i]
                    tgt_label = unit_labels_permuted[j]
                    print(f"  {i} ({src_label}) → {j} ({tgt_label})")

    # finally plot
    plot_graph_scc(mem, stim)
    plot_adj_scc(avgS, mem, stim)

print("✅ SCC‐only final plots complete.")
